In [ ]:
import numpy as np
from tqdm import tqdm

from load_data import load_dataset

from visuals.accrej_plots import get_mean_accs
from visuals.store_results import write_results_to_csv

from experiments_NBC import standard_experiment_NBC
from experiments_GeFs import standard_experiment_GeFs

This is the notebook that was used to run the full experiments for the PGM paper of 2026, with the used seeds and the used datasets.
The notebook produces files in the results_PGM folder, which are then used to create the figures in the paper.

In [ ]:
seeds = [0, 1]

for seed in seeds:
	np.random.seed(seed)  # for reproducibility

	print(f"\n--------------- Running experiment with seed: {seed} ---------------")

	do_NBC = True
	do_GeFs = True

	# parameters for the GeFs model
	n_estimators = 50
	max_iter = 100
	precision_bisearch_GeFs = 1e-9

	n_splits = 5 # number of split for cross-validation in NBC
	N_bootstraps = 10 # number of bootstrap models for calculating the information-theoretic uncertainty metrics
	n_runs_experiment = 5

	fraction_test = 0.4  # fraction of the data to use as a test set if no test set is provided
	max_size_test_set = 500  # maximum size of the test set if no test set is provided

	sets_for_experiments = ['solar_small', 'Australian', 'student_math_bin', 'student_port_bin', 'german', 'Healthy_aging', 'solar_big', 'bank', 'adult', 'BCW', 'spect_heart']

	##### Choose reliability metric(s) to compute/compare and some additional parameters ######
	uncertainty_metrics = ['max_prob', 'entropies', 'aleatoric', 'epistemic', 'total', 'margin_of_confidence']
	robustness_metrics = ['local_eps', 'chi2', 'total_var']

	metrics_to_compute = uncertainty_metrics + robustness_metrics


	for set_name in sets_for_experiments:
		print(f"\n--------------- {set_name} ---------------")

		X, y, X_test, y_test, C, F = load_dataset(set_name, randomize=True)
		if X_test is None:
			N = int(X.shape[0] * fraction_test)
		else:
			N = int(X_test.shape[0])

		print(f" with {C} classes, {F} features and {N} test points \n")

		# make a dictionary to store the results for this dataset
		list_of_dicts_NBC_classif = []
		list_of_dicts_NBC_robs = []
		list_of_dicts_GeFs_classif = []
		list_of_dicts_GeFs_robs = []

		for index_run in tqdm(range(n_runs_experiment)):

			X, y, X_test, y_test, C, F = load_dataset(set_name, randomize=True)
			X_train = None

			# if no test set is provided, split the data into a test and train set
			if X_test is None:
				test_set_size = int(fraction_test * X.shape[0])

				if test_set_size > max_size_test_set:
					test_set_size = max_size_test_set

				# random ordering of the data
				perm = np.random.permutation(len(y))
				X, y = X[perm], y[perm]

				X_test = X[:test_set_size]
				y_test = y[:test_set_size]

				X_train = X[test_set_size:]
				y_train = y[test_set_size:]

			elif X_train is None:
				X_train = X
				y_train = y

				if X_test.shape[0] > max_size_test_set:
					X_test = X_test[:max_size_test_set]
					y_test = y_test[:max_size_test_set]
			else:
				X_test = X_test[:max_size_test_set]
				y_test = y_test[:max_size_test_set]



			### NBC ###
			if do_NBC:
				rob_metrics, classification_results, y_pred = standard_experiment_NBC(X_train, y_train, X_test, y_test, C, F, metrics_to_compute, n_splits=n_splits, N_bootstraps=N_bootstraps)

				list_of_dicts_NBC_robs.append(rob_metrics)
				list_of_dicts_NBC_classif.append(classification_results)

			if do_GeFs:
				# ### GeFs ###
				rob_metrics, classification_results, y_pred = standard_experiment_GeFs(X_train, y_train, X_test, y_test, C, F, metrics_to_compute, n_estimators=n_estimators, N_bootstraps=N_bootstraps, precision_bisearch=precision_bisearch_GeFs, max_iter=max_iter)

				list_of_dicts_GeFs_robs.append(rob_metrics)
				list_of_dicts_GeFs_classif.append(classification_results)



		# name to use when storing the results
		file_name = f"{set_name}_{seed}"

		if do_NBC:
			list_of_rob_classif_results_NBC = np.vstack(list_of_dicts_NBC_classif)
			accuracies = np.mean(list_of_dicts_NBC_classif, axis=1)
			print(f'Accuracy NBC: {np.mean(accuracies)} ± {np.std(accuracies)}')

			list_of_rob_metrics_NBC_cp = list_of_dicts_NBC_robs.copy()
			write_results_to_csv(list_of_rob_classif_results_NBC, list_of_rob_metrics_NBC_cp, list_of_robm=True, set_name="_NBC", folder='results_PGM/csv')

			list_of_classif_results_NBC = list_of_dicts_NBC_classif
			list_of_rob_metrics_NBC = list_of_dicts_NBC_robs

		if do_GeFs:
			list_of_rob_classif_results_GeFs = np.vstack(list_of_dicts_GeFs_classif)
			accuracies = np.mean(list_of_dicts_GeFs_classif, axis=1)
			print(f'Accuracy GeFs: {np.mean(accuracies)} ± {np.std(accuracies)}')

			list_of_rob_metrics_GeFs_cp = list_of_dicts_GeFs_robs.copy()
			write_results_to_csv(list_of_rob_classif_results_GeFs, list_of_rob_metrics_GeFs_cp, list_of_robm=True, set_name="_GeFs", folder='results_PGM/csv')

			list_of_classif_results_GeFs = list_of_dicts_GeFs_classif
			list_of_rob_metrics_GeFs = list_of_dicts_GeFs_robs


		all_aucs_NBC = []
		all_aucs_GeFs = []

		for metric in metrics_to_compute:

			if do_NBC:
				accs_fixed_metric_NBC, auc_fixed_metric_NBC, stddevs_fixed_metric_NBC, stddevs_auc_fixed_metric_NBC = get_mean_accs(list_of_rob_metrics_NBC, list_of_classif_results_NBC, metric=metric, stddev=True)
				all_aucs_NBC.append((auc_fixed_metric_NBC, metric, stddevs_auc_fixed_metric_NBC))

				filename = f'results_PGM/accaccs/{file_name}_NBC.dat'
				with open(filename, 'a') as f:
					f.write(f'fixed:{metric}\n')
					f.write(','.join([str(x) for x in accs_fixed_metric_NBC]) + '\n')

			if do_GeFs:
				accs_fixed_metric_GeFs, auc_fixed_metric_GeFs, stddevs_fixed_metric_GeFs, stddevs_auc_fixed_metric_GeFs = get_mean_accs(list_of_rob_metrics_GeFs, list_of_classif_results_GeFs, metric=metric, stddev=True)
				all_aucs_GeFs.append((auc_fixed_metric_GeFs, metric, stddevs_auc_fixed_metric_GeFs))

				filename = f'results_PGM/accaccs/{file_name}_GeFs.dat'
				with open(filename, 'a') as f:
					f.write(f'fixed:{metric}\n')
					f.write(','.join([str(x) for x in accs_fixed_metric_GeFs]) + '\n')


		# sort all_aucs descending on the AUC value
		if do_NBC:
			all_aucs_NBC.sort(key=lambda x: x[0], reverse=True)

			filename = f"results_PGM/auc_summary/{seed}_NBC.dat"
			with open(filename, 'a') as f:
				if set_name == sets_for_experiments[0]:
					f.write(f"For each dataset, this table lists the reliability metrics sorted by their AUC (shown in parentheses) from highest to lowest.\n")
				f.write(f"\n{set_name:10.10s}:\t")
				# write all metrics and corresponding AUC values next to each other in the same line
				f.write(f' | '.join([f'{metric:8.8s} ({auc:.5f}±{stddev:.5f})' for auc, metric, stddev in all_aucs_NBC]) + '\n')

		if do_GeFs:
			all_aucs_GeFs.sort(key=lambda x: x[0], reverse=True)

			filename = f"results_PGM/auc_summary/{seed}_GeFs.dat"
			with open(filename, 'a') as f:
				if set_name == sets_for_experiments[0]:
					f.write(f"For each dataset, this table lists the reliability metrics sorted by their AUC (shown in parentheses) from highest to lowest.\n")
				f.write(f"\n{set_name:10.10s}:\t")
				# write all metrics and corresponding AUC values next to each other in the same line
				f.write(f' | '.join([f'{metric:8.8s} ({auc:.5f}±{stddev:.5f})' for auc, metric, stddev in all_aucs_GeFs]) + '\n')


Make ARCs of the different datasets. Files are created to make latex plots (see folder results_PGM/latex_plots).

In [ ]:
#### ARCs ####

folder_file_to_read = 'results_PGM/accaccs/'

data_set_names = ['adult',
					'Australian',
					'bank',
					'BCW',
					'german',
					'Healthy_aging',
					'solar_big',
					'solar_small',
					'spect_heart',
					'student_math_bin',
					'student_port_bin']


numbers = ['_0', '_1']

model = '_NBC'
# model = '_GeFs'

file_extension = '.dat'


uncertainty_metrics = ['max_prob', 'entropies', 'aleatoric', 'epistemic', 'total', 'margin_of_confidence']
robustness_metrics = ['local_eps', 'total_var', 'chi2']

for data_set_name in data_set_names:
	rob_metrics_temp = {}

	for r, random_num in enumerate(numbers):
		file_to_read = folder_file_to_read + data_set_name + random_num + model + file_extension

		with open(file_to_read, 'r') as f:
			lines = f.readlines()

			store_next_line = False

			for line in lines:
				if 'fixed' in line.strip():
					metric = line.split(':')[1].strip()
					store_next_line = True
					continue

				if store_next_line:
					if r == 0:
						rob_metrics_temp[metric] = []

					rob_metrics_temp[metric].append((np.array([float(x) for x in line.strip().split(',')])))

					store_next_line = False

	rob_metrics_temp = {key: np.array(rob_metrics_temp[key]) for key in rob_metrics_temp.keys()}

	rob_metrics = {}

	for key in rob_metrics_temp.keys():
		rob_metrics[key] = np.mean(rob_metrics_temp[key], axis=0)


	folder_file_to_write = 'results_PGM/latex_plots/ARCs/csvs/'

	filename = folder_file_to_write + data_set_name + model + '_arc.csv'

	metrics = uncertainty_metrics + robustness_metrics

	with open(filename, 'w') as f:
		f.write('n'+','+','.join(metrics)+'\n')

		for i in range(len(rob_metrics['local_eps'])):
			line = str(i)
			for metric in metrics:
				line += ',' + str(rob_metrics[metric][i])
			f.write(line+'\n')

The next cell makes the correct files to make Tables 3 and 4 from the paper, again to be used in the latex plots. 

In [ ]:
#### combi mean and wins ARC values per rejection rate ####

folder_file_to_read = 'results_PGM/accaccs/'

data_set_names = ['adult',
					'Australian',
					'bank',
					'BCW',
					'german',
					'Healthy_aging',
					'solar_big',
					'solar_small',
					'spect_heart',
					'student_math_bin',
					'student_port_bin']

rejection_rates = [0.1, 0.25, 0.75, 0.9]


numbers = ['_0', '_1']

nbc_model = False

if nbc_model:
	model_architecture = '_NBC'
else:
	model_architecture = '_GeFs'

file_extension = '.dat'

uncertainty_metrics = ['total', 'aleatoric', 'epistemic', 'max_prob', 'entropies', 'margin_of_confidence']
robustness_metrics = ['chi2', 'total_var', 'local_eps']
metrics = np.array(robustness_metrics + uncertainty_metrics)

mean_at_rr = [[] for _ in range(len(rejection_rates))]
maxes_at_rr = [[] for _ in range(len(rejection_rates))]

for d_i, data_set_name in enumerate(data_set_names):
	rob_metrics_temp = {}


	for r, random_num in enumerate(numbers):

		file_to_read = folder_file_to_read + data_set_name + random_num + model_architecture + file_extension

		with open(file_to_read, 'r') as f:
			lines = f.readlines()

			store_next_line = False

			for line in lines:
				if 'fixed' in line.strip():
					metric = line.split(':')[1].strip()
					store_next_line = True
					continue

				if store_next_line:
					if r == 0:
						rob_metrics_temp[metric] = []

					rob_metrics_temp[metric].append((np.array([float(x) for x in line.strip().split(',')])))

					store_next_line = False

	rob_metrics_temp = {key: np.array(rob_metrics_temp[key]) for key in rob_metrics_temp.keys()}

	rob_metrics = {}

	for key in rob_metrics_temp.keys():
		rob_metrics[key] = np.mean(rob_metrics_temp[key], axis=0)


	N = len(rob_metrics['local_eps'])
	rejection_rate_indices = [int(N * rr) for rr in rejection_rates]

	for i, rr in enumerate(rejection_rates):

		values_at_rr = []
		for metric in metrics:
			values_at_rr.append(rob_metrics[metric][rejection_rate_indices[i]])

		max_indices = np.where(values_at_rr == np.max(values_at_rr))[0]

		maxes_at_rr[i]+= list(metrics[max_indices])

		values_at_rr = np.array(values_at_rr)
		mean_at_rr[i].append(values_at_rr)


mean_at_rr = np.array(mean_at_rr)
mean_at_rr = np.mean(mean_at_rr, axis=1)


folder_file_to_write = 'results_PGM/latex_plots/RR_ARC/csvs/'

filename = folder_file_to_write + f"table_rr_combi{model_architecture}.csv"

with open(filename, 'w') as f:
	f.write('dataset'+','+',,'.join(metrics)+','+'\n')
	f.write(''+','+','.join(['\%', 'W']*len(metrics))+'\n')

for i, rr in enumerate(rejection_rates):
	# print(f"Rejection rate: {rr:.2f}")
	maxes_at_rr[i] = np.array(maxes_at_rr[i])
	uniques = np.unique(maxes_at_rr[i], return_counts=True)
	max_metrics = uniques[0]
	number_of_wins = uniques[1]


	with open(filename, 'a') as f:
		line = f"{rr*100:.0f}\%"
		for j, metric in enumerate(metrics):
			line += f",{mean_at_rr[i][j]*100:.2f}"

			if metric in max_metrics:
				line += f",{number_of_wins[np.where(max_metrics == metric)][0]}"
			else:
				line += f",0"

		f.write(line+'\n')


	# zip the unique metrics and their counts into a dictionary
	metric_counts = list(zip(uniques[0], uniques[1]))
	sorted_metric_counts = sorted(metric_counts, key=lambda x: x[1], reverse=True)
	# print(sorted_metric_counts, '\n')




folder_file_to_read = 'results_PGM/auc_summary/'
# folder_file_to_read = 'results_DS/auc/'


file_extension = '.dat'

data_set_dicts = {}

list_of_metrics =robustness_metrics + uncertainty_metrics

precision = 5

datasets = {}

aucs_per_metric = {}

for i, random_num in enumerate(numbers):
	file_to_read = folder_file_to_read + random_num + model_architecture + file_extension

	with open(file_to_read, 'r') as f:
		lines = f.readlines()

		for line in lines[2:]:
			if line == '\n':
				continue
			setn = line[:10].strip()
			# aucs = []
			aucs = {}

			metrics = line.split(':')[1].strip().split('|')

			best_unc = False
			for metric in metrics:
				metric_name = metric.split('(')[0].strip()

				if metric_name not in aucs_per_metric.keys():
					aucs_per_metric[metric_name] = []

				# print(metric_name[:8])

				if metric_name[:8] in robustness_metrics:
					auc = metric.split('(')[1].split("±")[0].strip()
					# aucs.append((metric_name, auc))
					if i == 0:
						aucs[metric_name] = float(auc)
					else:
						datasets[setn][metric_name] += float(auc)
						datasets[setn][metric_name] /= 2
					aucs_per_metric[metric_name].append(float(auc))

				elif not best_unc:
					auc = metric.split('(')[1].split("±")[0].strip()
					# aucs.append((metric_name, auc))
					best_unc = True
					if i == 0:
						aucs['uncertainty'] = float(auc)
					else:
						# print(datasets)
						datasets[setn]['uncertainty'] += float(auc)
						datasets[setn]['uncertainty'] /= 2
					aucs_per_metric[metric_name].append(float(auc))

				else:
					auc = metric.split('(')[1].split("±")[0].strip()
					aucs_per_metric[metric_name].append(float(auc))
					continue

			# print(metrics)
			if i == 0:
				datasets[setn] = aucs
			# print(line[:10].strip())

mean_aucs_per_metric = {m: np.mean(aucs_per_metric[m]) for m in aucs_per_metric.keys()}
# print(mean_aucs_per_metric)

mean_aucs = [(mean_aucs_per_metric[m],m) for m in aucs_per_metric.keys()]

# sort list of tuples by mean auc, first element of tuple, in descending order
mean_aucs.sort(reverse=True)


rejection_rates = [0.1, 0.25, 0.75, 0.9]
folder_file_to_write = 'results_PGM/latex_plots/RR_ARC/csvs/'

filename = folder_file_to_write + f"table_rr_combi{model_architecture}.csv"

with open(filename, 'a') as f:
	line = f"{{\\footnotesize AU-ARC}}"
	for metric in list_of_metrics:
		# make metric max 8 characters long
		metric_short = metric[:8]
		line += f",\\SetCell[c=2]{{c}} {mean_aucs_per_metric[metric]:.5f},"
	f.write(line+'\n')

	# zip the unique metrics and their counts into a dictionary
	metric_counts = list(zip(uniques[0], uniques[1]))
	sorted_metric_counts = sorted(metric_counts, key=lambda x: x[1], reverse=True)